In [1]:
import torch
from torch import nn

class ManualLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

        self.gamma = nn.Parameter(
            torch.ones(normalized_shape)
        )
        self.beta = nn.Parameter(
            torch.zeros(normalized_shape)
        )
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)

        var = x.var(
            dim=-1,
            keepdim=True,
            unbiased=False
        )

        x_norm = (x - mean) / torch.sqrt(var + self.eps)

        return self.gamma * x_norm + self.beta

In [3]:
torch.manual_seed(42)

x = torch.randn(3, 4)

manual_ln = ManualLayerNorm(4)
torch_ln = nn.LayerNorm(4)

# 确保两者参数完全相同
with torch.no_grad():
    torch_ln.weight.copy_(manual_ln.gamma)
    torch_ln.bias.copy_(manual_ln.beta)

manual_output = manual_ln(x)
torch_output = torch_ln(x)

print("Manual output:\n", manual_output)
print("PyTorch output:\n", torch_output)

torch.testing.assert_close(
    manual_output,
    torch_output,
    rtol=1e-5,
    atol=1e-6
)

print("Mean:", manual_output.mean(dim=-1))
print("Variance:", manual_output.var(dim=-1, unbiased=False))

Manual output:
 tensor([[ 1.4150, -1.4102,  0.0257, -0.0305],
        [-0.9276, -0.1964,  1.6731, -0.5491],
        [-0.2914, -1.2908,  0.0853,  1.4969]], grad_fn=<AddBackward0>)
PyTorch output:
 tensor([[ 1.4150, -1.4102,  0.0257, -0.0305],
        [-0.9276, -0.1964,  1.6731, -0.5491],
        [-0.2914, -1.2908,  0.0853,  1.4969]],
       grad_fn=<NativeLayerNormBackward0>)
Mean: tensor([-9.3132e-10, -1.4901e-08,  1.7881e-07], grad_fn=<MeanBackward1>)
Variance: tensor([0.9982, 1.0000, 0.9997], grad_fn=<VarBackward0>)


In [4]:
batch_a = torch.tensor([
    [1., 2., 3., 4.],
    [2., 3., 4., 5.],
    [3., 4., 5., 6.],
    [4., 5., 6., 7.]
])

batch_b = torch.tensor([
    [1., 2., 3., 4.],
    [100., 100., 100., 100.],
    [200., 200., 200., 200.],
    [300., 300., 300., 300.]
])

ln = nn.LayerNorm(4)

bn = nn.BatchNorm1d(
    num_features=4,
    affine=False,
    track_running_stats=False
)

ln.train()
bn.train()

ln_a = ln(batch_a)
ln_b = ln(batch_b)

bn_a = bn(batch_a)
bn_b = bn(batch_b)

print("LayerNorm, batch A:", ln_a[0])
print("LayerNorm, batch B:", ln_b[0])

print("BatchNorm, batch A:", bn_a[0])
print("BatchNorm, batch B:", bn_b[0])

# LayerNorm 只看自己的特征，不受其他样本影响
torch.testing.assert_close(ln_a[0], ln_b[0])

# BatchNorm 训练时依赖整个 batch 的统计量
assert not torch.allclose(bn_a[0], bn_b[0])

LayerNorm, batch A: tensor([-1.3416, -0.4472,  0.4472,  1.3416], grad_fn=<SelectBackward0>)
LayerNorm, batch B: tensor([-1.3416, -0.4472,  0.4472,  1.3416], grad_fn=<SelectBackward0>)
BatchNorm, batch A: tensor([-1.3416, -1.3416, -1.3416, -1.3416])
BatchNorm, batch B: tensor([-1.3389, -1.3362, -1.3335, -1.3307])
